# Skynode auto-train

Drop phone clips into **Drive ▸ skynode ▸ phone_videos**, then run the cells below. Each run:
1. skips clips it has already used,
2. labels the new ones with the current model and trains on them,
3. replaces the model **only if the new one is at least as good** on held-out clips,
4. saves a `.onnx` to **Drive ▸ skynode ▸ models** when it replaces it.

Runtime ▸ Change runtime type ▸ **T4 GPU** first. Name clips `drone_*`, `aircraft_*` or `sky_*` when they show one thing.

In [ ]:
PROJECT = "/content/drive/MyDrive/skynode"
REVIEW = False          # True = stop after labelling so you can check the boxes first
EPOCHS = 25

from google.colab import drive
drive.mount("/content/drive")
import os
if not os.path.exists("/content/Skynode"):
    !git clone -q --depth 1 -b main https://github.com/bsalsa2/Skynode /content/Skynode
else:
    !git -C /content/Skynode pull -q
!pip install -q ultralytics

In [ ]:
flag = " --review" if REVIEW else ""
!python /content/Skynode/training/auto_train.py --project "{PROJECT}" --epochs {EPOCHS}{flag}

### Reading the result
- `"status": "promoted"`: a new `skynode-auto-<date>.onnx` is in **models**. Copy it to the brain (see `brain/config.v3.toml`).
- `"status": "kept old model"`: the new one wasn't better. Nothing is lost: the clips stay used, so add more clips and run again.
- `"status": "no frames"` or `No new clips`: nothing to do yet.

Honest limit: unless you use `REVIEW = True` and check the boxes, the scores compare the models on the old model's own labels, which makes the old model look stronger. The gate is therefore cautious, not perfect.